# BoW Setup

Getting the data ready for Naive Bayes + Decision Trees. Mostly copying the cleaning from the shared data setup notebook, then saving the balanced dataframes as CSVs so I don't have to re-run the Kaggle downloads every time.

Three dataframes I'm saving:
- `balanced_df` — all songs, random downsample to 20% hit rate
- `balanced_2000_df` — random downsample, 2000 songs per decade, 20% hit rate (this is what BERT uses)
- `balanced_1700_unPOP_df` — only keeps popularity-0 non-hits, 1700/decade, 20% hit rate

## Data setup

Copied from the shared `dataSetup_hitPrediction_NLP.ipynb`.

In [ ]:
!pip install opendatasets -q

import opendatasets as od
od.download('https://www.kaggle.com/datasets/serkantysz/550k-spotify-songs-audio-lyrics-and-genres?select=songs.csv')
od.download('https://www.kaggle.com/datasets/dhruvildave/billboard-the-hot-100-songs')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: kanguz
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/serkantysz/550k-spotify-songs-audio-lyrics-and-genres


100%|██████████| 235M/235M [00:01<00:00, 220MB/s]



Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: kanguz
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/dhruvildave/billboard-the-hot-100-songs


100%|██████████| 3.05M/3.05M [00:00<00:00, 109MB/s]


In [ ]:
import pandas as pd
import re

#loading datasets from kaggle
songs_df   = pd.read_csv('/content/550k-spotify-songs-audio-lyrics-and-genres/songs.csv')
charts_df  = pd.read_csv('/content/billboard-the-hot-100-songs/charts.csv')

print('songs_df columns:', songs_df.columns.tolist())
print('songs_df shape:  ', songs_df.shape)

songs_df columns: ['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres']
songs_df shape:   (550622, 24)


In [ ]:
# songs per decade
songs_df['decade'] = (songs_df['year'] // 10 * 10).astype(str) + 's'
print(songs_df.groupby('decade').size().to_string())

decade
1900s        30
1910s         3
1920s       156
1930s       288
1940s       437
1950s      4119
1960s     11344
1970s     17570
1980s     20729
1990s     42913
2000s    162161
2010s    225362
2020s     65510


In [ ]:
import ast

def normalize_text(text):
    if pd.isna(text): return ''
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

#removing the featuere title in artist lists
def clean_artist_name(artist):
    for sep in [' featuring ', ' feat. ', ' feat ', ' ft. ', ' ft ', ' x ', ' & ', ' with ']:
        artist = artist.lower().split(sep)[0]
    return artist.strip()

# some songs have multiple artists listed as a stringified list, we need to extract the primary artist for matching
def get_primary_artist(artists_str):
    try:
        artists_list = ast.literal_eval(artists_str)
        return normalize_text(artists_list[0])
    except:
        return clean_artist_name(normalize_text(str(artists_str)))

# normalizing our datasets
songs_df['song_normalized']   = songs_df['name'].apply(normalize_text)
songs_df['artist_normalized'] = songs_df['artists'].apply(get_primary_artist)

charts_df['song_normalized']   = charts_df['song'].apply(normalize_text)
charts_df['artist_normalized'] = charts_df['artist'].apply(lambda x: clean_artist_name(normalize_text(x)))

In [ ]:
# creating the is_hit label feature to songs dataframe
# a song is a hit if it ever reached top 10, matched on song + artist name
top10 = charts_df[charts_df['rank'] <= 10][['song_normalized', 'artist_normalized']].drop_duplicates()
top10['is_hit'] = 1

songs_df = songs_df.merge(top10, on=['song_normalized', 'artist_normalized'], how='left')
songs_df['is_hit'] = songs_df['is_hit'].fillna(0).astype(int)

print(f"Total songs:  {len(songs_df):,}")
print(f"Hits (top10): {songs_df['is_hit'].sum():,}")
print(f"Overall hit rate: {songs_df['is_hit'].mean():.1%}")

Total songs:  550,622
Hits (top10): 7,292
Overall hit rate: 1.3%


In [ ]:
# checking how many hits we have per decade in a percentage (not a lot!)
decade_stats = songs_df.groupby('decade')['is_hit'].agg(
    total='count',
    hits='sum'
)
decade_stats['hit_rate'] = (decade_stats['hits'] / decade_stats['total']).map('{:.1%}'.format)
print(decade_stats[decade_stats.index >= '1960s'].to_string())

         total  hits hit_rate
decade                       
1960s    11344   437     3.9%
1970s    17570   604     3.4%
1980s    20729   796     3.8%
1990s    42913   850     2.0%
2000s   162161  1821     1.1%
2010s   225362  2128     0.9%
2020s    65510   620     0.9%


In [ ]:
# this dataset only uses non-hits that are popularity 0 metric

print(songs_df.columns.tolist())
songs_df_unPop = songs_df[~((songs_df['popularity'] > 0) & (songs_df['is_hit'] == 0))]

['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres', 'decade', 'song_normalized', 'artist_normalized', 'is_hit']


In [ ]:
# checking how many hits we have per decade in a percentage after taking out non-zero non_hit songs
decade_stats = songs_df_unPop.groupby('decade')['is_hit'].agg(
    total='count',
    hits='sum'
)
decade_stats['hit_rate'] = (decade_stats['hits'] / decade_stats['total']).map('{:.1%}'.format)
print(decade_stats[decade_stats.index >= '1960s'].to_string())

        total  hits hit_rate
decade                      
1960s    1798   437    24.3%
1970s    3496   604    17.3%
1980s    4212   796    18.9%
1990s   10536   850     8.1%
2000s   43777  1821     4.2%
2010s   70817  2128     3.0%
2020s   18636   620     3.3%


In [ ]:
# @title Downsample 1700 — popularity-0 non-hits only
# 20% hit rate, 1700 songs per decade
balanced_2000_decades = []

for decade, group in songs_df_unPop.groupby('decade'):
    if decade < '1960s':
        continue
    hits = group[group['is_hit'] == 1]
    non_hits = group[group['is_hit'] == 0]

    # x4 for 20% hit rate
    target_hits = 340
    target_non_hits = 340 * 4
    hits_sampled = hits.sample(n=target_hits, random_state=42)
    non_hits_sampled = non_hits.sample(n=target_non_hits, random_state=42)

    balanced_2000 = pd.concat([hits_sampled, non_hits_sampled])
    balanced_2000_decades.append(balanced_2000)
    print(f"{decade}: {len(hits_sampled)} hits, {len(non_hits_sampled)} non-hits, hit rate: {len(hits_sampled)/len(balanced_2000):.1%}")

balanced_1700_unPOP_df = pd.concat(balanced_2000_decades).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"\nTotal: {len(balanced_1700_unPOP_df):,} songs")

1960s: 340 hits, 1360 non-hits, hit rate: 20.0%
1970s: 340 hits, 1360 non-hits, hit rate: 20.0%
1980s: 340 hits, 1360 non-hits, hit rate: 20.0%
1990s: 340 hits, 1360 non-hits, hit rate: 20.0%
2000s: 340 hits, 1360 non-hits, hit rate: 20.0%
2010s: 340 hits, 1360 non-hits, hit rate: 20.0%
2020s: 340 hits, 1360 non-hits, hit rate: 20.0%

Total: 11,900 songs


In [ ]:
# @title Downsample 2000 — random
# 20% hit rate, 2000 songs per decade
balanced_2000_decades = []

for decade, group in songs_df.groupby('decade'):
    if decade < '1960s':
        continue
    hits = group[group['is_hit'] == 1]
    non_hits = group[group['is_hit'] == 0]

    # x4 for 20% hit rate
    target_hits = 400
    target_non_hits = 400 * 4
    hits_sampled = hits.sample(n=target_hits, random_state=42)
    non_hits_sampled = non_hits.sample(n=target_non_hits, random_state=42)

    balanced_2000 = pd.concat([hits_sampled, non_hits_sampled])
    balanced_2000_decades.append(balanced_2000)
    print(f"{decade}: {len(hits_sampled)} hits, {len(non_hits_sampled)} non-hits, hit rate: {len(hits_sampled)/len(balanced_2000):.1%}")

balanced_2000_df = pd.concat(balanced_2000_decades).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"\nTotal: {len(balanced_2000_df):,} songs")

1960s: 400 hits, 1600 non-hits, hit rate: 20.0%
1970s: 400 hits, 1600 non-hits, hit rate: 20.0%
1980s: 400 hits, 1600 non-hits, hit rate: 20.0%
1990s: 400 hits, 1600 non-hits, hit rate: 20.0%
2000s: 400 hits, 1600 non-hits, hit rate: 20.0%
2010s: 400 hits, 1600 non-hits, hit rate: 20.0%
2020s: 400 hits, 1600 non-hits, hit rate: 20.0%

Total: 14,000 songs


In [ ]:
# @title Downsample Full
# 20% hit rate, all available songs per decade
balanced_decades = []

for decade, group in songs_df.groupby('decade'):
    if decade < '1960s':
        continue
    hits = group[group['is_hit'] == 1]
    non_hits = group[group['is_hit'] == 0]

    # x4 for 20% hit rate
    target_non_hits = min(len(hits) * 4, len(non_hits))
    non_hits_sampled = non_hits.sample(n=target_non_hits, random_state=42)

    balanced = pd.concat([hits, non_hits_sampled])
    balanced_decades.append(balanced)
    print(f"{decade}: {len(hits)} hits, {len(non_hits_sampled)} non-hits, hit rate: {len(hits)/len(balanced):.1%}")

balanced_df = pd.concat(balanced_decades).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"\nTotal: {len(balanced_df):,} songs")

1960s: 437 hits, 1748 non-hits, hit rate: 20.0%
1970s: 604 hits, 2416 non-hits, hit rate: 20.0%
1980s: 796 hits, 3184 non-hits, hit rate: 20.0%
1990s: 850 hits, 3400 non-hits, hit rate: 20.0%
2000s: 1821 hits, 7284 non-hits, hit rate: 20.0%
2010s: 2128 hits, 8512 non-hits, hit rate: 20.0%
2020s: 620 hits, 2480 non-hits, hit rate: 20.0%

Total: 36,280 songs


In [ ]:
#removing the unecessary columns from our datasets.
balanced_2000_df = balanced_2000_df[['id', 'song_normalized', 'artist_normalized', 'lyrics', 'is_hit', 'decade']]
balanced_df = balanced_df[['id', 'song_normalized', 'artist_normalized', 'lyrics', 'is_hit', 'decade']]
balanced_1700_unPOP_df = balanced_1700_unPOP_df[['id', 'song_normalized', 'artist_normalized', 'lyrics', 'is_hit', 'decade']]

print(balanced_2000_df.columns.tolist())

['id', 'song_normalized', 'artist_normalized', 'lyrics', 'is_hit', 'decade']


## Save dataframes

Dumping all three to CSV so the NB + DT notebook can load them directly.

In [ ]:
balanced_df.to_csv('balanced_df.csv', index=False)
balanced_2000_df.to_csv('balanced_2000_df.csv', index=False)
balanced_1700_unPOP_df.to_csv('balanced_1700_unPOP_df.csv', index=False)

print('Saved:')
print(f'  balanced_df.csv            — {len(balanced_df):,} rows')
print(f'  balanced_2000_df.csv       — {len(balanced_2000_df):,} rows')
print(f'  balanced_1700_unPOP_df.csv — {len(balanced_1700_unPOP_df):,} rows')

Saved:
  balanced_df.csv            — 36,280 rows
  balanced_2000_df.csv       — 14,000 rows
  balanced_1700_unPOP_df.csv — 11,900 rows


## BoW sanity check

Quick check that `CountVectorizer` is producing something reasonable before I move to training.

Config:
- `stop_words='english'` — drop the fillers
- `min_df=2` — ignore words that only show up in one song
- `ngram_range=(1, 2)` — unigrams + bigrams, so stuff like "baby baby" and "let go" gets picked up too



In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

df = balanced_2000_df[['lyrics', 'is_hit']].dropna(subset=['lyrics']).copy()
X = df['lyrics'].astype(str).values

vectorizer = CountVectorizer(
    stop_words='english',
    min_df=2,
    ngram_range=(1, 2)
)
X_bow = vectorizer.fit_transform(X)

print(f'Documents:  {X_bow.shape[0]:,}')
print(f'Vocab size: {X_bow.shape[1]:,}')
print(f'Non-zero entries: {X_bow.nnz:,}')

# top terms — should look like normal lyric words, not junk
import numpy as np
term_totals = np.asarray(X_bow.sum(axis=0)).flatten()
vocab = vectorizer.get_feature_names_out()
top_idx = term_totals.argsort()[-20:][::-1]
print('\nTop 20 most frequent tokens:')
for i in top_idx:
    print(f'  {vocab[i]:<30} {term_totals[i]:,}')

Documents:  14,000
Vocab size: 144,884
Non-zero entries: 1,328,720

Top 20 most frequent tokens:
  love                           27,580
  oh                             25,049
  don                            23,202
  know                           21,204
  just                           19,215
  like                           18,332
  yeah                           17,041
  ll                             16,548
  got                            15,790
  baby                           15,163
  time                           11,751
  let                            11,736
  ve                             11,735
  come                           10,946
  want                           9,911
  say                            9,446
  way                            9,404
  make                           9,285
  cause                          8,255
  gonna                          8,146
